# 01-02 生徒AI設計・学習曲線確認ノートブック

最終更新: 2026-07-23 10:49:24 JST

一次方程式の生徒AIについて、内部パラメータの設計と理解度-正答率の関係を確認するNotebookです。

このNotebookは研究全体の一部です。単独でも動きますが、論文用には以下の順番で確認します。

```text
00 paper_core_experiment.ipynb       : 論文用の最小実験整理
01 student_ai_colab.ipynb            : 生徒AIの設計確認
02 student_ai_colab.ipynb            : 理解度と正答率の学習曲線確認
03 personality_experiment.ipynb      : 個人特徴と伝達AI分類
04 teaching_strategy_experiment.ipynb: 複数生徒クラスと授業方略
```


<!-- notebook-role-guide -->
## このNotebookで確認すること

- 生徒状態JSONの構造を確認する
- 理解度、誤概念、個人特徴が回答にどう影響するかを見る
- 理解度0-100と正答率の学習曲線を確認する

## 実行順

- Colabでは 1-3 を先に実行して環境を作る
- 設計確認は 5-10 を実行する
- 学習曲線は 12-15 を実行する
- Git更新やテストは 17-18 を必要な時だけ実行する

## 主な出力

- 生徒AIの内部状態
- 1問ごとの回答
- 理解度と正答率の表・グラフ
- 誤答傾向の確認結果

## 注意

- ColabでGit更新セルを実行しても、すでに開いているNotebook画面のセル内容は自動では書き換わりません。
- Notebook自体を更新した場合は、GitHub上の最新版Notebookを開き直してください。
- LLMを使うセルはロードに時間がかかります。まずはmockまたは軽量モデルで確認してください。


## 1. Clone repository

`REPO_URL` は設定済みです。すでに `/content/student-ai` が存在する場合はcloneをスキップします。

In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/Hiromu-0219/student-ai-test.git"
PROJECT_DIR = Path("/content/student-ai")

if not PROJECT_DIR.exists():
    !git clone {REPO_URL} {PROJECT_DIR}

%cd /content/student-ai

## 2. Pull latest code from GitHub

毎回Colabを使い始めるときは、このセルを実行してGitHubの最新版を取り込みます。初回clone直後に実行しても問題ありません。

注意: `student_ai_colab.ipynb` 自体のセル追加・移動は、すでに開いているColab画面には自動反映されないことがあります。その場合はGitHubから最新版のノートブックを開き直してください。

In [ ]:
%cd /content/student-ai
!git status --short --branch
!git pull
print('GitHub update check finished.')

## 3. Install dependencies

In [ ]:
!pip install -q -r requirements.txt

## 4. Optional Hugging Face login

Gemma系モデルやprivate/gated modelを使う場合に実行してください。

In [ ]:
# from huggingface_hub import login
# login()

## 5. Smoke test with mock model

モデルをダウンロードせず、状態管理・回答生成・ログ保存の経路を確認します。

In [ ]:
from src.student_ai import StudentAISimulator

sim = StudentAISimulator(use_mock_model=True)
result = sim.answer(
    student_id="S001",
    problem="2x + 3 = 11 を解いてください。",
)

print(result["answer"])

## 6. Model parameters

ここでモデルID、4bit量子化、生成パラメータを調整します。

- `MODEL_ID`: 使用するHugging FaceモデルID
- `load_in_4bit`: GPUメモリ節約。Colabでは基本 `True`
- `compute_dtype`: 計算精度。まず `bfloat16`、動かない場合は `float16`
- `max_new_tokens`: 生徒回答の最大長
- `temperature`: 高いほど回答が揺れる
- `top_p`: 低いほど候補語を絞る
- `repetition_penalty`: 同じ表現の繰り返しを抑える

In [ ]:
from src.config import GenerationConfig, ModelLoadConfig

# 使用するモデル。Gemma 3 4Bを使うなら "google/gemma-3-4b-it" などに変更します。
MODEL_ID = "Qwen/Qwen3-4B"

MODEL_LOAD_CONFIG = ModelLoadConfig(
    load_in_4bit=True,  # True: 4bit量子化でメモリ節約。Colabでは基本True
    bnb_4bit_quant_type="nf4",  # 通常はnf4
    bnb_4bit_use_double_quant=True,  # True: さらにメモリ節約
    compute_dtype="bfloat16",  # 動かないGPUではfloat16も試す
)

GENERATION_CONFIG = GenerationConfig(
    max_new_tokens=256,  # 長い返答が必要なら増やす。短い生徒回答なら128-256
    temperature=0.7,  # 低いほど安定、高いほど生徒らしい揺れが出る
    top_p=0.9,  # 低いほど保守的。通常は0.8-0.95
    do_sample=True,  # True: 毎回少し変わる。False: 安定寄り
    repetition_penalty=1.05,  # 同じ語句の繰り返し抑制
)

## 7. Student parameters

ここで使う生徒IDと、生徒AIの状態パラメータを編集します。実行すると `data/students/{STUDENT_ID}.json` に保存されます。

知識状態は `0-100` です。心理・性格系は `very_low / low / medium / high / very_high` の5段階です。

- `score`: 一次方程式全体の総合理解度
- `can_solve_ax_plus_b_equals_c`: `ax + b = c` 型を解く力
- `can_transpose_terms`: 移項の理解
- `can_divide_by_coefficient`: 係数で割る理解
- `can_handle_negative_numbers`: 負の数への対応
- `can_handle_fractions`: 分数への対応
- `misconceptions`: 誤概念。誤答や迷いとして発話に反映される
- `learning_speed`: 将来、授業後に知識スコアを更新するときの伸びやすさ
- `self_efficacy`: 自己効力感。自信の強さ
- `question_tendency`: 質問しやすさ
- `motivation`: 学習意欲
- `big_five`: 性格傾向。発話量、丁寧さ、不安の出方などに影響

In [ ]:
from src.state_manager import StateManager

STUDENT_ID = "S002"

state_manager = StateManager("data/students")
student_state = state_manager.load_student(STUDENT_ID)

# ここを編集すると、テスト時の正答/誤答方針と発話が変わります。
student_state["knowledge_state"] = {
    "linear_equation": {
        "level": "low",  # scoreから自動更新されます
        "score": 25,  # 一次方程式全体の理解度。0=未理解、100=十分理解
        "can_solve_ax_plus_b_equals_c": 25,  # ax + b = c 型を解く力
        "can_transpose_terms": 5,  # 移項で符号を変える理解
        "can_divide_by_coefficient": 20,  # 3x=15 で3で割る理解
        "can_handle_negative_numbers": 5,  # マイナスを含む計算への対応
        "can_handle_fractions": 5,  # 分数を含む式への対応
    }
}
student_state["misconceptions"] = [
    "移項は項を反対側へ動かすだけで符号はそのままだと思っている",  # 誤概念1
    "3x = 15 のような式で、3を引けばよいと考えることがある",  # 誤概念2
]
student_state["learning_speed"] = "low"  # 将来の授業更新での伸びやすさ
student_state["big_five"] = {
    "openness": "medium",  # 新しい説明や別解を受け入れやすいか
    "conscientiousness": "low",  # 丁寧に途中式を書くか
    "extraversion": "medium",  # 発話量や反応の積極性
    "agreeableness": "high",  # 教師に素直に合わせる傾向
    "neuroticism": "high",  # 不安や焦りの出やすさ
}
student_state["self_efficacy"] = "low"  # 自分は解けると思える度合い
student_state["question_tendency"] = "high"  # わからない時に質問する傾向
student_state["motivation"] = "medium"  # 学習意欲

state_manager.save_student(student_state)
student_state

## 8. Create simulator

学力テスト検証で使う生徒AIを作ります。現在は、認知モデルによる内部妥当性の確認を優先するため、標準では `USE_MOCK_MODEL=True` にしています。LLM発話を確認したい場合だけ `False` に変更してください。

In [ ]:
from src.student_ai import StudentAISimulator

USE_MOCK_MODEL = True  # True: LLMを読まずに内部妥当性を確認 / False: Qwenなどの実LLMで発話確認

sim = StudentAISimulator(
    model_id=MODEL_ID,
    load_in_4bit=MODEL_LOAD_CONFIG.load_in_4bit,
    model_load_config=MODEL_LOAD_CONFIG,
    generation_config=GENERATION_CONFIG,
    use_mock_model=USE_MOCK_MODEL,
)

print(f"student_id={STUDENT_ID}, model={sim.agent.model_id}")

## 9. Future extension: Interactive lesson disabled

現在の検証では授業対話は使いません。将来、授業による状態変化を検証するときに有効化する想定です。

In [ ]:
print("現在の検証では授業対話セルは使用しません。")
print("将来使う場合は sim.respond(STUDENT_ID, teacher_message) を使います。")

## 10. Optional smoke check: one-shot answer

任意の軽い確認用です。知識更新はしない設定で1問だけ解かせます。

In [ ]:
result = sim.respond(
    STUDENT_ID,
    "3x - 5 = 10 を解いてください。",
    update_knowledge=False,
)

print(result["answer"])

## 11. Future extension: controlled learning intervention disabled

現在の検証では学習介入は使いません。将来、授業後の状態変化を検証するときに有効化する想定です。

In [ ]:
print("現在の検証では学習介入セルは使用しません。")
print("将来使う場合は sim.apply_learning_intervention(...) を使います。")

## 12. Assessment test

生徒AIに学力テストを受けさせます。テストは授業中の発話ではなく、問題に対する答えだけを出す形式です。このテストは測定用なので、`knowledge_state` は更新しません。結果は `data/assessments/` に保存されます。

In [ ]:
from src.test_runner import TestRunner

TEST_ID = "linear_equation_basic_001"

runner = TestRunner(simulator=sim)
assessment_result = runner.run_test(
    student_id=STUDENT_ID,
    test_id=TEST_ID,
)

print("テスト:", assessment_result["title"])
print("得点:", assessment_result["score_percentage"], "%")
print("正答数:", assessment_result["correct_count"], "/", assessment_result["total_count"])
print("スキル別スコア:")
for skill, score in assessment_result["skill_scores"].items():
    print("-", skill, score, "%")

評価ログを確認します。

In [ ]:
from pathlib import Path

print(Path("data/assessments/human_readable.md").read_text(encoding="utf-8")[-2000:])

## 13. Understanding score vs accuracy graph

理解度スコアと20問テストの正答率の相関を確認します。理解度 `0,20,40,60,80,100` の検証用生徒を作り、同じ20問テストを受けさせて散布図を描きます。

- `TEST_ID_FOR_CORRELATION` は20問テストです。
- `update_knowledge=False` なので、テスト中に知識スコアは更新しません。
- テスト時は認知モデルが `knowledge_state` から正答/誤答方針と `target_answer` を決めます。
- 正答率の採点には `target_answer` を使い、LLMの生出力は `raw_student_answer` として保存します。
- 同じ問題には全理解度で同じ判定ロールを使うため、理解度が上がるほど段階的に誤答が減りやすくなります。
- 誤概念ペナルティと誤概念文は低理解度ほど強く、高理解度ほど弱くなります。
- 現在の検証では、理解度パラメータを変えたときに正答率がどう変わるかを見ます。
- グラフに加えて、問題ごとの正誤表も表示・CSV保存します。
- 正答数もsummary表に残しますが、グラフの縦軸は正答率です。
- LLMで確認する場合も、相関実験の採点は認知モデル側の制御回答を使います。
- mock/LLMに関係なく、相関実験の正答率は認知モデル側の制御回答で計算します。

In [ ]:
%%script false --no-raise
# LLM発話確認セルは一時停止中です。再開する場合はこの行を削除してください。
from copy import deepcopy
import math
import matplotlib.pyplot as plt
import pandas as pd

from src.state_manager import StateManager
from src.test_runner import TestRunner

TEST_ID_FOR_CORRELATION = "linear_equation_20q_001"
UNDERSTANDING_SCORES = [0, 20, 40, 60, 80, 100]

def score_to_level(score):
    if score < 20:
        return "very_low"
    if score < 40:
        return "low"
    if score < 60:
        return "medium"
    if score < 80:
        return "high"
    return "very_high"

def misconceptions_by_score(score):
    if score >= 90:
        return []
    if score >= 70:
        return [
            "移項時に符号変更をたまに迷うが、ほぼ修正できる",
            "係数で割る場面で一瞬迷うが、説明があれば修正できる",
        ]
    if score >= 40:
        return [
            "移項では符号を変えることを理解し始めているが、まだ混乱することがある",
            "3x = 15 では係数で割る必要を理解し始めているが、引き算と混同することがある",
        ]
    return [
        "移項は項を反対側へ動かすだけで符号はそのままだと思っている",
        "3x = 15 のような式で、3を引けばよいと考えることがある",
    ]

state_manager = StateManager("data/students")
base_state = state_manager.load_student(STUDENT_ID)

runner = TestRunner(simulator=sim)
correlation_rows = []
answer_rows = []

for score in UNDERSTANDING_SCORES:
    profile_id = f"CORR_SCORE_{score:03d}"
    state = deepcopy(base_state)
    state["student_id"] = profile_id
    state["name"] = profile_id
    state["learning_history"] = []
    state["knowledge_state"] = {
        "linear_equation": {
            "level": score_to_level(score),
            "score": score,
            "can_solve_ax_plus_b_equals_c": score,
            "can_transpose_terms": score,
            "can_divide_by_coefficient": score,
            "can_handle_negative_numbers": score,
            "can_handle_fractions": score,
        }
    }
    state["misconceptions"] = misconceptions_by_score(score)
    state["self_efficacy"] = score_to_level(score)
    state["motivation"] = "medium"
    state["question_tendency"] = "high" if score < 40 else "medium"
    state_manager.save_student(state)

    result = runner.run_test(student_id=profile_id, test_id=TEST_ID_FOR_CORRELATION)
    correlation_rows.append({
        "student_id": profile_id,
        "understanding_score": score,
        "accuracy": result["score_percentage"],
        "correct_count": result["correct_count"],
        "total_count": result["total_count"],
    })
    for question_result in result["question_results"]:
        directive = question_result.get("assessment_directive", {})
        grading = question_result["grading"]
        answer_rows.append({
            "student_id": profile_id,
            "understanding_score": score,
            "question_id": question_result["question_id"],
            "skill": question_result["skill"],
            "difficulty": question_result["difficulty"],
            "expected_answer": question_result["expected_answer"],
            "student_value": grading["student_value"],
            "controlled_answer": question_result["student_answer"],
            "raw_student_answer": question_result.get("raw_student_answer"),
            "is_correct": grading["is_correct"],
            "mark": "○" if grading["is_correct"] else "×",
            "target_correct": directive.get("target_correct"),
            "correct_probability": directive.get("correct_probability"),
            "misconception_penalty": directive.get("misconception_penalty"),
            "misconception_strength": directive.get("misconception_strength"),
            "active_misconceptions": directive.get("active_misconceptions"),
        })
    print(profile_id, "理解度=", score, "正答数=", result["correct_count"], "/", result["total_count"], "正答率=", result["score_percentage"], "%")

xs = [row["understanding_score"] for row in correlation_rows]
ys = [row["accuracy"] for row in correlation_rows]
x_mean = sum(xs) / len(xs)
y_mean = sum(ys) / len(ys)
numerator = sum((x - x_mean) * (y - y_mean) for x, y in zip(xs, ys))
den_x = math.sqrt(sum((x - x_mean) ** 2 for x in xs))
den_y = math.sqrt(sum((y - y_mean) ** 2 for y in ys))
correlation = numerator / (den_x * den_y) if den_x and den_y else 0

plt.figure(figsize=(7, 5))
plt.scatter(xs, ys)
plt.plot(xs, ys, linestyle="--", alpha=0.6)
for row in correlation_rows:
    plt.annotate(row["student_id"], (row["understanding_score"], row["accuracy"]), fontsize=8)
plt.title(f"Understanding score vs test accuracy (r={correlation:.2f})")
plt.xlabel("Knowledge score before test (0-100)")
plt.ylabel("Test accuracy (%)")
plt.xlim(-5, 105)
plt.ylim(-5, 105)
plt.grid(True, alpha=0.3)
plt.show()

correlation_df = pd.DataFrame(correlation_rows)
answer_df = pd.DataFrame(answer_rows)
correctness_table = answer_df.pivot(
    index="question_id",
    columns="understanding_score",
    values="mark",
)

correlation_df.to_csv("data/assessments/understanding_accuracy_summary.csv", index=False)
answer_df.to_csv("data/assessments/understanding_accuracy_detail.csv", index=False)
correctness_table.to_csv("data/assessments/understanding_accuracy_correctness_table.csv")

print("正誤表: 行=問題, 列=理解度スコア")
display(correctness_table)
print("詳細表")
display(answer_df)
print("CSV saved under data/assessments/")

correlation_df

## 14. Dense understanding sweep, 5-point intervals

理解度 0,5,10,...,100 の21段階で、大量の自動生成問題を使って正答率を確認します。

- このセルではLLM発話を生成せず、認知モデルだけで正答/誤答を決めます。
- GENERATED_QUESTION_COUNT で問題数を調整できます。標準は500問です。
- 問題数を増やすほど正答率のブレが小さくなります。Colabで重い場合は200程度に下げてください。
- 画面には正誤表の先頭50問だけ表示し、全件はCSVに保存します。

In [ ]:
from copy import deepcopy
from fractions import Fraction
import math
import matplotlib.pyplot as plt
import pandas as pd

from src.cognitive_model import CognitiveModel
from src.grader import LinearEquationGrader
from src.state_manager import StateManager

UNDERSTANDING_SCORES_DENSE = list(range(0, 101, 5))
GENERATED_QUESTION_COUNT = 500  # Colabで重い場合は200、余裕があれば1000以上に変更

SKILLS = [
    "can_solve_ax_plus_b_equals_c",
    "can_transpose_terms",
    "can_divide_by_coefficient",
    "can_handle_negative_numbers",
    "can_handle_fractions",
]

def score_to_level(score):
    if score < 20:
        return "very_low"
    if score < 40:
        return "low"
    if score < 60:
        return "medium"
    if score < 80:
        return "high"
    return "very_high"

def misconceptions_by_score(score):
    if score >= 90:
        return []
    if score >= 70:
        return [
            "移項時に符号変更をたまに迷うが、ほぼ修正できる",
            "係数で割る場面で一瞬迷うが、説明があれば修正できる",
        ]
    if score >= 40:
        return [
            "移項では符号を変えることを理解し始めているが、まだ混乱することがある",
            "3x = 15 では係数で割る必要を理解し始めているが、引き算と混同することがある",
        ]
    return [
        "移項は項を反対側へ動かすだけで符号はそのままだと思っている",
        "3x = 15 のような式で、3を引けばよいと考えることがある",
    ]

def format_value(value):
    value = Fraction(value)
    if value.denominator == 1:
        return str(value.numerator)
    return f"{value.numerator}/{value.denominator}"

def make_question(index):
    skill = SKILLS[index % len(SKILLS)]
    x = (index % 21) - 10
    if x == 0:
        x = 3
    difficulty = 1

    if skill == "can_solve_ax_plus_b_equals_c":
        a = (index % 8) + 2
        b = (index % 11) - 5
        c = a * x + b
        problem = f"{a}x {'+' if b >= 0 else '-'} {abs(b)} = {c} を解きなさい。"
        difficulty = 2
    elif skill == "can_transpose_terms":
        b = (index % 17) - 8
        c = x + b
        problem = f"x {'+' if b >= 0 else '-'} {abs(b)} = {c} を解きなさい。"
        difficulty = 1
    elif skill == "can_divide_by_coefficient":
        a = (index % 9) + 2
        c = a * x
        problem = f"{a}x = {c} を解きなさい。"
        difficulty = 1
    elif skill == "can_handle_negative_numbers":
        a = -((index % 7) + 2)
        b = (index % 13) - 6
        c = a * x + b
        problem = f"{a}x {'+' if b >= 0 else '-'} {abs(b)} = {c} を解きなさい。"
        difficulty = 3
    else:
        d = (index % 5) + 2
        b = (index % 9) - 4
        c = Fraction(x, d) + b
        problem = f"x/{d} {'+' if b >= 0 else '-'} {abs(b)} = {format_value(c)} を解きなさい。"
        difficulty = 3

    return {
        "question_id": f"QGEN_{index + 1:04d}_{skill}",
        "problem": problem,
        "answer": f"x = {format_value(x)}",
        "skill": skill,
        "difficulty": difficulty,
    }

questions = [make_question(index) for index in range(GENERATED_QUESTION_COUNT)]
state_manager = StateManager("data/students")
base_state = state_manager.load_student(STUDENT_ID)
cognitive_model = CognitiveModel()
grader = LinearEquationGrader()
correlation_rows = []
answer_rows = []

for score in UNDERSTANDING_SCORES_DENSE:
    profile_id = f"DENSE_SCORE_{score:03d}"
    state = deepcopy(base_state)
    state["student_id"] = profile_id
    state["name"] = profile_id
    state["learning_history"] = []
    state["knowledge_state"] = {
        "linear_equation": {
            "level": score_to_level(score),
            "score": score,
            "can_solve_ax_plus_b_equals_c": score,
            "can_transpose_terms": score,
            "can_divide_by_coefficient": score,
            "can_handle_negative_numbers": score,
            "can_handle_fractions": score,
        }
    }
    state["misconceptions"] = misconceptions_by_score(score)
    state["self_efficacy"] = score_to_level(score)
    state["motivation"] = "medium"
    state["question_tendency"] = "high" if score < 40 else "medium"
    state_manager.save_student(state)

    correct_count = 0
    for question in questions:
        directive = cognitive_model.build_assessment_directive(
            student_state=state,
            question=question,
        )
        controlled_answer = directive["target_answer"]
        grading = grader.grade(question["answer"], controlled_answer)
        correct_count += grading["score"]
        answer_rows.append({
            "student_id": profile_id,
            "understanding_score": score,
            "question_id": question["question_id"],
            "skill": question["skill"],
            "difficulty": question["difficulty"],
            "expected_answer": question["answer"],
            "controlled_answer": controlled_answer,
            "student_value": grading["student_value"],
            "is_correct": grading["is_correct"],
            "mark": "○" if grading["is_correct"] else "×",
            "target_correct": directive.get("target_correct"),
            "correct_probability": directive.get("correct_probability"),
            "roll": directive.get("roll"),
            "misconception_penalty": directive.get("misconception_penalty"),
            "misconception_strength": directive.get("misconception_strength"),
            "active_misconceptions": directive.get("active_misconceptions"),
        })

    accuracy = round((correct_count / len(questions)) * 100, 1)
    correlation_rows.append({
        "student_id": profile_id,
        "understanding_score": score,
        "accuracy": accuracy,
        "correct_count": correct_count,
        "total_count": len(questions),
    })
    print(profile_id, "理解度=", score, "正答数=", correct_count, "/", len(questions), "正答率=", accuracy, "%")

xs = [row["understanding_score"] for row in correlation_rows]
ys = [row["accuracy"] for row in correlation_rows]
x_mean = sum(xs) / len(xs)
y_mean = sum(ys) / len(ys)
numerator = sum((x - x_mean) * (y - y_mean) for x, y in zip(xs, ys))
den_x = math.sqrt(sum((x - x_mean) ** 2 for x in xs))
den_y = math.sqrt(sum((y - y_mean) ** 2 for y in ys))
correlation = numerator / (den_x * den_y) if den_x and den_y else 0

plt.figure(figsize=(9, 5))
plt.scatter(xs, ys)
plt.plot(xs, ys, linestyle="--", alpha=0.6)
plt.title(f"Understanding score vs test accuracy ({len(questions)} questions, r={correlation:.2f})")
plt.xlabel("Knowledge score before test (0-100)")
plt.ylabel("Test accuracy (%)")
plt.xlim(-2, 102)
plt.ylim(-5, 105)
plt.grid(True, alpha=0.3)
plt.show()

correlation_df = pd.DataFrame(correlation_rows)
answer_df = pd.DataFrame(answer_rows)
correctness_table = answer_df.pivot(
    index="question_id",
    columns="understanding_score",
    values="mark",
)

correlation_df.to_csv("data/assessments/dense_understanding_accuracy_summary.csv", index=False)
answer_df.to_csv("data/assessments/dense_understanding_accuracy_detail.csv", index=False)
correctness_table.to_csv("data/assessments/dense_understanding_accuracy_correctness_table.csv")

print("正誤表 preview: 行=問題, 列=理解度スコア。全件はCSVに保存しています。")
display(correctness_table.head(50))
print("詳細表 preview")
display(answer_df.head(100))
print("CSV saved under data/assessments/")

correlation_df

## 15. Parameter error tendency check

パラメータによって誤答傾向が変わるかを確認します。同じ問題を、異なる生徒状態の検証用プロファイルに解かせ、回答と正誤を比較します。

- 元の `S001/S002/S003` は変更しません。
- 検証用に `PARAM_*` の生徒JSONを作ります。
- `update_knowledge=False` なので、この検証では知識スコアは更新しません。
- 標準では次のLLM発話確認セルは実行しない設定にしています。
- LLMに発話させる確認をしたい場合だけ、別セルとして `RUN_LLM_UTTERANCE_CHECK=True` のガード付きコードを追加してください。
- LLMで試す場合は `USE_MOCK_MODEL=False` の `sim` を先に作ってください。

In [ ]:
from copy import deepcopy
from src.grader import LinearEquationGrader
from src.state_manager import StateManager

CHECK_PROBLEM = "3x - 5 = 10 を解きなさい。"
EXPECTED_ANSWER = "x = 5"
TRIALS_PER_PROFILE = 3  # LLMの揺れを見るため、同じ条件で複数回試します

state_manager = StateManager("data/students")
base_state = state_manager.load_student("S002")
grader = LinearEquationGrader()

profiles = {
    "PARAM_LOW_KNOWLEDGE": {
        "description": "知識が低く、移項の誤概念が強い生徒",
        "knowledge_state": {
            "linear_equation": {
                "level": "very_low",
                "score": 10,
                "can_solve_ax_plus_b_equals_c": 15,
                "can_transpose_terms": 0,
                "can_divide_by_coefficient": 10,
                "can_handle_negative_numbers": 0,
                "can_handle_fractions": 0,
            }
        },
        "misconceptions": [
            "移項しても符号は変えなくてよいと思っている",
            "3x = 15 では3を引けばよいと思っている",
        ],
        "self_efficacy": "very_low",
        "question_tendency": "high",
        "motivation": "medium",
    },
    "PARAM_HIGH_ANXIETY": {
        "description": "知識は中程度だが不安が強く、迷いが出やすい生徒",
        "knowledge_state": {
            "linear_equation": {
                "level": "medium",
                "score": 55,
                "can_solve_ax_plus_b_equals_c": 60,
                "can_transpose_terms": 45,
                "can_divide_by_coefficient": 60,
                "can_handle_negative_numbers": 35,
                "can_handle_fractions": 25,
            }
        },
        "misconceptions": ["移項時の符号変化に自信がない"],
        "self_efficacy": "low",
        "question_tendency": "very_high",
        "motivation": "medium",
        "big_five": {
            "openness": "medium",
            "conscientiousness": "medium",
            "extraversion": "low",
            "agreeableness": "high",
            "neuroticism": "very_high",
        },
    },
    "PARAM_HIGH_KNOWLEDGE": {
        "description": "知識が高く、誤概念がほぼない生徒",
        "knowledge_state": {
            "linear_equation": {
                "level": "very_high",
                "score": 90,
                "can_solve_ax_plus_b_equals_c": 95,
                "can_transpose_terms": 90,
                "can_divide_by_coefficient": 90,
                "can_handle_negative_numbers": 85,
                "can_handle_fractions": 75,
            }
        },
        "misconceptions": [],
        "self_efficacy": "high",
        "question_tendency": "low",
        "motivation": "high",
    },
}

for profile_id, overrides in profiles.items():
    state = deepcopy(base_state)
    state["student_id"] = profile_id
    state["name"] = profile_id
    state["learning_history"] = []
    description = overrides.pop("description")
    state.update(overrides)
    state_manager.save_student(state)
    overrides["description"] = description

results = []
for profile_id, profile in profiles.items():
    for trial in range(1, TRIALS_PER_PROFILE + 1):
        response = sim.respond(profile_id, CHECK_PROBLEM, update_knowledge=False)
        grading = grader.grade(EXPECTED_ANSWER, response["answer"])
        results.append({
            "profile_id": profile_id,
            "description": profile["description"],
            "trial": trial,
            "is_correct": grading["is_correct"],
            "student_value": grading["student_value"],
            "answer": response["answer"],
        })

for item in results:
    mark = "OK" if item["is_correct"] else "NG"
    print(f"[{mark}] {item['profile_id']} trial={item['trial']} student_value={item['student_value']}")
    print("設定:", item["description"])
    print("回答:", item["answer"])
    print("-" * 80)

## 16. Check lesson logs

In [ ]:
from pathlib import Path

print(Path("data/logs/human_readable.md").read_text(encoding="utf-8")[-2000:])

## 17. Optional: reclone repository

pullで衝突したり、Colab上の作業内容を捨ててGitHubの最新版に戻したい場合だけ使います。通常の順番実行ではスキップしてください。

In [ ]:
# 必要な場合だけ実行
# %cd /content
# !rm -rf student-ai
# !git clone {REPO_URL} /content/student-ai
# %cd /content/student-ai
# !pip install -q -r requirements.txt

## 18. Optional tests

標準テストはmock modelのみを使うため、LLMのダウンロードなしで実行できます。

In [ ]:
!python -m pytest

<!-- student-ai-expanded-evaluation -->
## 19. Expanded student AI evaluation

生徒AI単体の制御妥当性をまとめて確認します。

このセルでは、次を一度に出します。

- 理解度と正答率の学習曲線
- 誤概念あり/なしの比較
- スキル別の弱点比較
- 個人特徴を変えた発話サンプル

結果は `data/assessments/student_ai_evaluation_summary.txt` に保存されます。


In [ ]:
from pprint import pprint

import importlib
import src.experiment.student_ai_evaluation as student_ai_evaluation_module

importlib.reload(student_ai_evaluation_module)

run_student_ai_evaluation = student_ai_evaluation_module.run_student_ai_evaluation
export_student_ai_evaluation = student_ai_evaluation_module.export_student_ai_evaluation
export_student_ai_evaluation_for_codex = student_ai_evaluation_module.export_student_ai_evaluation_for_codex

student_ai_evaluation = run_student_ai_evaluation(
    student_id=STUDENT_ID,
    test_id="linear_equation_20q_001",
    understanding_levels=list(range(0, 101, 10)),
    use_mock_model=USE_MOCK_MODEL,
)

summary_path = export_student_ai_evaluation(student_ai_evaluation)
codex_path = export_student_ai_evaluation_for_codex(student_ai_evaluation)

print("summary_path:", summary_path)
print("codex_share_path:", codex_path)
print("\nsummary:")
pprint(student_ai_evaluation["summary"])

print("\nlearning_curve:")
for row in student_ai_evaluation["learning_curve"]:
    print(row)

print("\nutterance_samples:")
for sample in student_ai_evaluation["utterance_samples"]:
    print(f"- {sample['profile_id']}: {sample['utterance']}")


<!-- cognitive-model-comparison -->
## 20. Cognitive model comparison

Compare the legacy cognitive model and the BKT/IRT-inspired cognitive model under the same test condition.

- legacy: previous understanding-centered model
- bkt_irt: uses skill mastery, item difficulty, guess, and slip
- This cell does not call an LLM.
- Share file: data/assessments/cognitive_model_comparison_for_codex.txt


In [ ]:
from pprint import pprint

import importlib
import pandas as pd
import matplotlib.pyplot as plt

import src.experiment.student_ai_evaluation as student_ai_evaluation_module

importlib.reload(student_ai_evaluation_module)

compare_cognitive_models = student_ai_evaluation_module.compare_cognitive_models
export_cognitive_model_comparison_for_codex = student_ai_evaluation_module.export_cognitive_model_comparison_for_codex

cognitive_model_comparison = compare_cognitive_models(
    student_id=STUDENT_ID,
    test_id="linear_equation_20q_001",
    understanding_levels=list(range(0, 101, 10)),
    use_mock_model=True,
)

comparison_path = export_cognitive_model_comparison_for_codex(cognitive_model_comparison)
comparison_df = pd.DataFrame(cognitive_model_comparison["learning_curve_comparison"])

print("comparison_share_path:", comparison_path)
print("summary:")
pprint(cognitive_model_comparison["summary"])

display(comparison_df)

plt.figure(figsize=(8, 5))
plt.plot(
    comparison_df["understanding"],
    comparison_df["legacy_average_correct_probability"],
    marker="o",
    label="legacy probability",
)
plt.plot(
    comparison_df["understanding"],
    comparison_df["bkt_irt_average_correct_probability"],
    marker="o",
    label="bkt_irt probability",
)
plt.xlabel("understanding / skill mastery")
plt.ylabel("average correct probability")
plt.ylim(0, 100)
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()
